# 02 · Train FHE-friendly CNN
Train the shallow CNN on FER2013 with Square ($x^2$) activations and strided convolution (no pooling layers).
Architecture: Conv2d(stride=3) -> Square -> Flatten -> FC -> Square -> FC.
This architecture is designed to be FHE-friendly by minimizing multiplicative depth.


### 블록 1 · 라이브러리/모델 불러오기
학습에 필요한 PyTorch, NumPy, tqdm, 그리고 FHE 전용 CNN 모듈을 임포트합니다.


In [34]:
import sys
from pathlib import Path

try:
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
print(f'Python path prepared with project root: {PROJECT_ROOT}')


Python path prepared with project root: /Users/hyunwookkim/Documents/study/25y2s/보안프로젝트설계/프로젝트/프로토타입/learning_test_01/fhe_emotion


In [35]:
import json
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
from tqdm.notebook import tqdm

from models.fhe_cnn import FHEEmotionCNN, extract_fhe_parameters

### 블록 2 · 경로 및 하이퍼파라미터 정의
데이터 위치, 저장 경로, 배치 크기와 에폭 수 등을 설정합니다.


In [36]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_OUT = PROJECT_ROOT / 'models' / 'fhe_cnn_fer2013.pt'
NORM_STATS_PATH = PROJECT_ROOT / 'models' / 'normalization_stats.json'
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


Device: cpu


### 블록 3 · 전처리된 텐서 로딩
데이터 준비 노트북에서 저장한 이미지·레이블·클래스 가중치 텐서를 불러옵니다.


In [37]:
def load_tensor(name: str) -> torch.Tensor:
    path = DATA_DIR / f'{name}.pt'
    tensor = torch.load(path)
    print(f'Loaded {name} -> {tensor.shape}')
    return tensor

train_images = load_tensor('train_images')
val_images = load_tensor('val_images')
test_images = load_tensor('test_images')
train_labels = load_tensor('train_labels')
val_labels = load_tensor('val_labels')
test_labels = load_tensor('test_labels')
class_weights = load_tensor('class_weights')


Loaded train_images -> torch.Size([28709, 1, 48, 48])
Loaded val_images -> torch.Size([3589, 1, 48, 48])
Loaded test_images -> torch.Size([3589, 1, 48, 48])
Loaded train_labels -> torch.Size([28709])
Loaded val_labels -> torch.Size([3589])
Loaded test_labels -> torch.Size([3589])
Loaded class_weights -> torch.Size([7])


### 블록 4 · 정규화 통계 계산
학습 세트의 평균과 표준편차를 구해 JSON으로 저장하고 이후 노멀라이즈에 사용합니다.


In [38]:
train_mean = train_images.mean().item()
train_std = train_images.std().item()
print(f'Train mean: {train_mean:.4f}, std: {train_std:.4f}')
stats = {'mean': train_mean, 'std': train_std}
with open(NORM_STATS_PATH, 'w') as f:
    json.dump(stats, f, indent=2)
print('Saved normalization stats ->', NORM_STATS_PATH)


Train mean: 0.5072, std: 0.2550
Saved normalization stats -> /Users/hyunwookkim/Documents/study/25y2s/보안프로젝트설계/프로젝트/프로토타입/learning_test_01/fhe_emotion/models/normalization_stats.json


### 블록 5 · 변환 및 데이터셋 구성
데이터 증강(Flip, Crop, Rotation) 파이프라인과 PyTorch Dataset/DataLoader를 정의합니다.


In [39]:
base_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[train_mean], std=[train_std]),
])
train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.RandomResizedCrop(size=48, scale=(0.9, 1.0)),
    T.RandomRotation(10),
    base_transform,
])
eval_transform = T.Compose([
    T.ToPILImage(),
    base_transform,
])

class AugmentedFERDataset(Dataset):
    def __init__(self, images: torch.Tensor, labels: torch.Tensor, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        lbl = self.labels[idx]
        array = img.squeeze(0).numpy().astype(np.float32)
        if self.transform:
            img_tensor = self.transform(array)
        else:
            img_tensor = torch.tensor(array)[None, :, :]
            img_tensor = T.Normalize(mean=[train_mean], std=[train_std])(img_tensor)
        return img_tensor, lbl

train_dataset = AugmentedFERDataset(train_images, train_labels, transform=train_transform)
val_dataset = AugmentedFERDataset(val_images, val_labels, transform=eval_transform)
test_dataset = AugmentedFERDataset(test_images, test_labels, transform=eval_transform)
NUM_WORKERS = 0  # 노트북 환경에서는 multi-processing pickle 이슈 방지를 위해 0으로 둔다.
PIN_MEMORY = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)



### 블록 6 · 모델 및 최적화 기법 설정
`FHEEmotionCNN`, 가중치가 적용된 CrossEntropyLoss, Adam 옵티마이저를 초기화합니다.


In [40]:
model = FHEEmotionCNN().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
best_val_acc = 0.0
history = []


In [41]:
# Verify model architecture and output shape
print(model)
dummy_input = torch.randn(1, 1, 48, 48).to(device)
with torch.no_grad():
    output = model(dummy_input)
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
assert output.shape == (1, 7), f"Expected output shape (1, 7), got {output.shape}"


FHEEmotionCNN(
  (conv1): Conv2d(1, 4, kernel_size=(7, 7), stride=(3, 3))
  (act1): Square()
  (fc1): Linear(in_features=784, out_features=64, bias=True)
  (act2): Square()
  (fc2): Linear(in_features=64, out_features=7, bias=True)
)
Input shape: torch.Size([1, 1, 48, 48])
Output shape: torch.Size([1, 7])


### 블록 7 · 학습 루프
에폭별로 학습/검증 손실·정확도를 계산하며 최적 모델을 저장합니다.


In [42]:
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    total = 0
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch} / {EPOCHS}'):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        total += images.size(0)
    train_loss /= total
    train_acc = train_correct / total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)
    val_loss /= val_total
    val_acc = val_correct / val_total
    history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc, 'val_loss': val_loss, 'val_acc': val_acc})
    print(f'Epoch {epoch}: train_loss={train_loss:.4f} train_acc={train_acc:.3f} | val_loss={val_loss:.4f} val_acc={val_acc:.3f}')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_OUT)
        print('Saved new best model ->', MODEL_OUT)

print('Training complete. Best val acc:', best_val_acc)


Epoch 1 / 50:   0%|          | 0/449 [00:00<?, ?it/s]

Epoch 1: train_loss=1.8658 train_acc=0.258 | val_loss=1.8015 val_acc=0.327
Saved new best model -> /Users/hyunwookkim/Documents/study/25y2s/보안프로젝트설계/프로젝트/프로토타입/learning_test_01/fhe_emotion/models/fhe_cnn_fer2013.pt


Epoch 2 / 50:   0%|          | 0/449 [00:00<?, ?it/s]

Epoch 2: train_loss=1.7756 train_acc=0.330 | val_loss=1.7517 val_acc=0.363
Saved new best model -> /Users/hyunwookkim/Documents/study/25y2s/보안프로젝트설계/프로젝트/프로토타입/learning_test_01/fhe_emotion/models/fhe_cnn_fer2013.pt


Epoch 3 / 50:   0%|          | 0/449 [00:00<?, ?it/s]

...

Epoch 48: train_loss=1.3772 train_acc=0.452 | val_loss=1.5853 val_acc=0.421


Epoch 49 / 50:   0%|          | 0/449 [00:00<?, ?it/s]

Epoch 49: train_loss=1.3738 train_acc=0.455 | val_loss=1.6728 val_acc=0.430


Epoch 50 / 50:   0%|          | 0/449 [00:00<?, ?it/s]

Epoch 50: train_loss=1.3648 train_acc=0.452 | val_loss=1.6481 val_acc=0.440
Saved new best model -> /Users/hyunwookkim/Documents/study/25y2s/보안프로젝트설계/프로젝트/프로토타입/learning_test_01/fhe_emotion/models/fhe_cnn_fer2013.pt
Training complete. Best val acc: 0.43995541933686266


### 블록 8 · 테스트 평가
보존한 최적 가중치로 테스트 세트 정확도를 측정하고 히스토리를 출력합니다.


In [45]:
def evaluate(loader):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)
    return correct / total

test_acc = evaluate(test_loader)
print(f'Test accuracy: {test_acc:.3f}')
print('History:', history)


Test accuracy: 0.440
History: [{'epoch': 1, 'train_loss': 1.8657562695159455, 'train_acc': 0.25775889094012333, 'val_loss': 1.8014529149291176, 'val_acc': 0.3265533574811925}, {'epoch': 2, 'train_loss': 1.7756445527579154, 'train_acc': 0.32986171583823887, 'val_loss': 1.75172120125123, 'val_acc': 0.36305377542490946}, {'epoch': 3, 'train_loss': 1.7182607730107895, 'train_acc': 0.35689156710439235, 'val_loss': 1.6934875855707598, 'val_acc': 0.3616606297018668}, {'epoch': 4, 'train_loss': 1.679657934389583, 'train_acc': 0.3687693754571737, 'val_loss': 1.6806090256735355, 'val_acc': 0.39509612705488995}, {'epoch': 5, 'train_loss': 1.6431967640351617, 'train_acc': 0.3748998571876415, 'val_loss': 1.6521968642593792, 'val_acc': 0.3574811925327389}, {'epoch': 6, 'train_loss': 1.6230987800426204, 'train_acc': 0.3782785885959107, 'val_loss': 1.655008397616527, 'val_acc': 0.385065477848983}, {'epoch': 7, 'train_loss': 1.6019596687831983, 'train_acc': 0.3860810198892333, 'val_loss': 1.61360198553

In [44]:
# Verify FHE parameter extraction
print("Extracting FHE parameters...")
params = extract_fhe_parameters(model)
print("Keys:", params.keys())
print("Conv layers:", len(params['conv']))
print("Linear layers:", len(params['linear']))
for i, layer in enumerate(params['conv']):
    print(f"Conv[{i}] weight shape: {layer['weight'].shape}, bias shape: {layer['bias'].shape}")
for i, layer in enumerate(params['linear']):
    print(f"Linear[{i}] weight shape: {layer['weight'].shape}, bias shape: {layer['bias'].shape}")


Extracting FHE parameters...
Keys: dict_keys(['conv', 'linear'])
Conv layers: 1
Linear layers: 2
Conv[0] weight shape: torch.Size([4, 1, 7, 7]), bias shape: torch.Size([4])
Linear[0] weight shape: torch.Size([64, 784]), bias shape: torch.Size([64])
Linear[1] weight shape: torch.Size([7, 64]), bias shape: torch.Size([7])
